# 1. Configuración del Entorno y Carga de Componentes

## 1.1. Importación de Librerías

Importamos las librerías necesarias para el análisis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import scanpy as sc
import joblib

from scipy.sparse import csr_matrix

# Configuramos el estilo de las visualizaciones
sns.set_theme(style="whitegrid")
print("Librerías importadas correctamente.")

## 1.2. Definición de Rutas

Definimos las rutas a los datos de entrada y a las carpetas de salida.

In [ ]:
DATA_PROCESSED_PATH = '../data/processed/'
MODELS_PATH = '../outputs/models/'
FIGURES_PATH = '../outputs/figures/'

# Nombres de los ficheros de entrada
BULK_COUNTS_FILENAME = 'TCGA-LUAD_counts_for_deconvolution.parquet'
BULK_CLINICAL_FILENAME = 'TCGA-LUAD_clinical_for_deconvolution.parquet'
REF_SC_FILENAME = 'lung_cancer_processed_for_modeling.h5ad'
MODEL_FILENAME = 'mlp_marker_genes.joblib'
OPTIMIZED_SIGNATURE_FILENAME = 'optimized_signature_matrix.parquet'

os.makedirs(os.path.join(FIGURES_PATH, 'deconvolution'), exist_ok=True)

print("Rutas definidas.")

## 1.3. Carga de Datos de TCGA

Cargamos los datos de expresión y clínicos de la cohorte de TCGA-LUAD, que fueron procesados en el notebook anterior.

In [ ]:
print("Cargando datos de TCGA (bulk RNA-seq)...")
bulk_counts_df = pd.read_parquet(os.path.join(DATA_PROCESSED_PATH, BULK_COUNTS_FILENAME))
bulk_clinical_df = pd.read_parquet(os.path.join(DATA_PROCESSED_PATH, BULK_CLINICAL_FILENAME))

print("Datos de TCGA cargados:")
print(f"  - Matriz de conteos: {bulk_counts_df.shape[0]} muestras x {bulk_counts_df.shape[1]} genes")
print(f"  - Datos clínicos: {bulk_clinical_df.shape[0]} muestras x {bulk_clinical_df.shape[1]} variables")


## 1.4. Carga de Datos de Referencia

Cargamos el objeto AnnData de scRNA-seq que contiene los perfiles de expresión de referencia para cada tipo celular.

In [ ]:
print("\nCargando datos de referencia (scRNA-seq)...")
adata_ref = sc.read_h5ad(os.path.join(DATA_PROCESSED_PATH, REF_SC_FILENAME))

print("Datos de referencia cargados:")
print(adata_ref)

## 1.5. Carga de la Matriz de Firmas Optimizada

Cargamos la matriz de firmas genéticas que fue generada y optimizada en el Notebook 2. Esta matriz se basa en los 500 genes globalmente más discriminativos identificados por un modelo Random Forest.

In [ ]:
print("\nCargando la matriz de firmas optimizada...")
signature_matrix_optimized = pd.read_parquet(
    os.path.join(DATA_PROCESSED_PATH, OPTIMIZED_SIGNATURE_FILENAME)
)

print("Matriz de firmas optimizada cargada:")
print(f"Dimensiones: {signature_matrix_optimized.shape[0]} genes x {signature_matrix_optimized.shape[1]} tipos celulares")
cell_types = signature_matrix_optimized.columns.tolist()
print(f"Tipos celulares definidos: {cell_types}")
display(signature_matrix_optimized.head())


# 2. Carga y Análisis de Resultados de Deconvolución (EPIC)

## 2.1. Carga de los Resultados

Cargamos el fichero CSV generado por el script de R, que contiene las proporciones celulares estimadas para cada muestra por el método EPIC.


In [ ]:
print("--- Cargando los resultados de la deconvolución desde R ---")

# Ruta al fichero de resultados
DECONV_RESULTS_FILENAME_EPIC = 'deconv_results_epic.csv'
deconv_results_path = os.path.join(DATA_PROCESSED_PATH, DECONV_RESULTS_FILENAME_EPIC)

temp_df = pd.read_csv(deconv_results_path, index_col=0)

deconv_results_df = temp_df.T

# R reemplaza espacios con puntos
deconv_results_df.columns = deconv_results_df.columns.str.replace('.', ' ', regex=False)
# Nombramos el índice para ser consistentes
deconv_results_df.index.name = 'barcode'

print("\nResultados de la deconvolución cargados y transpuestos:")
print("Dimensiones finales:", deconv_results_df.shape)
display(deconv_results_df.head())


## 2.2. Fusión de Resultados con Datos Clínicos

Unimos las proporciones celulares estimadas con la tabla de datos clínicos para facilitar los análisis posteriores.

In [ ]:
print("--- Fusionando resultados de deconvolución con datos clínicos ---")

analysis_df = pd.concat([bulk_clinical_df, deconv_results_df], axis=1)

# Verificamos que no haya NaN en las proporciones y rellenamos si es necesario
proportion_cols = analysis_df.columns[-11:]
analysis_df[proportion_cols] = analysis_df[proportion_cols].fillna(0)

print("Fusión completada.")
display(analysis_df.head())

## 2.3. Composición Celular Promedio de la Cohorte

Para obtener una visión general, calculamos y visualizamos la proporción promedio de cada tipo celular en toda la cohorte de tumores.

In [ ]:
print("\n--- Visualizando la composición celular promedio (incluyendo otherCells) ---")

# Obtenemos los nombres de todas las fracciones celulares
all_cell_fractions = analysis_df.columns[-11:].tolist()

# Calculamos la media de cada columna de tipo celular
mean_proportions = analysis_df[all_cell_fractions].mean().sort_values(ascending=False)

plt.figure(figsize=(12, 7))
sns.barplot(x=mean_proportions.index, y=mean_proportions.values)
plt.title('Composición Celular Promedio (con otherCells) en TCGA-LUAD', fontsize=16)
plt.ylabel('Proporción Promedio')
plt.xlabel('Tipo Celular')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

display(mean_proportions.to_frame(name='Proporción Promedio'))
print("\nAnálisis: La fracción 'otherCells' representa, en promedio, "
      f"{mean_proportions.get('otherCells', 0):.2%} de la composición tisular, "
      "indicando una porción significativa de la señal de expresión que no es explicada por nuestra firma.")


### 2.3.1. Validación Cruzada con Proporciones de Referencia scRNA-seq

Para validar la plausibilidad de nuestras estimaciones de deconvolución, comparamos la composición celular promedio obtenida en la cohorte de TCGA con las proporciones celulares reales observadas en nuestro dataset de referencia de scRNA-seq. Aunque no se espera una coincidencia perfecta debido a diferencias tecnológicas y de cohorte, una correlación general en el ranking de abundancia de los tipos celulares aumentaría la confianza en nuestro método.

In [ ]:
print("\n--- Comparando las proporciones de EPIC con la referencia scRNA-seq ---")

# 1. Calcular las proporciones reales en el dataset de scRNA-seq de referencia
# Usamos el objeto `adata_ref` que cargamos al principio
sc_proportions = adata_ref.obs['cell_type'].value_counts(normalize=True)

# 2. Obtener las proporciones promedio de la deconvolución de EPIC
# Usamos .mean() sobre el DataFrame de resultados de la deconvolución
deconv_proportions = deconv_results_df.mean()

# 3. Crear un DataFrame combinado para facilitar la visualización
# Nos aseguramos de que ambos usen el mismo índice para la unión
comparison_df = pd.DataFrame({
    'Deconvolución (EPIC)': deconv_proportions,
    'Referencia (scRNA-seq)': sc_proportions
}).fillna(0).sort_values(by='Referencia (scRNA-seq)', ascending=False) # Ordenamos para un mejor gráfico

# 4. Visualizar la comparación
fig, axes = plt.subplots(1, 2, figsize=(20, 8))
fig.suptitle('Validación: Proporciones de Deconvolución (EPIC) vs. Referencia scRNA-seq', fontsize=18)

# Gráfico de barras agrupado
comparison_df.plot(kind='bar', ax=axes[0])
axes[0].set_title('Composición Promedio por Método', fontsize=14)
axes[0].set_ylabel('Proporción Promedio')
axes[0].set_xlabel('Tipo Celular')
axes[0].tick_params(axis='x', rotation=45)

# Scatter plot para ver la correlación
sns.regplot(
    data=comparison_df,
    x='Referencia (scRNA-seq)',
    y='Deconvolución (EPIC)',
    ax=axes[1]
)
# Añadir línea de identidad (y=x) para una comparación perfecta
max_val = comparison_df.max().max() * 1.1 # Un poco de margen
axes[1].plot([0, max_val], [0, max_val], 'r--', label='Identidad (y=x)')
axes[1].set_title('Correlación entre Proporciones Estimadas y Reales', fontsize=14)
axes[1].set_xlabel('Proporción en scRNA-seq (Observada)')
axes[1].set_ylabel('Proporción en Deconvolución (Estimada)')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# Mostramos la tabla para una inspección numérica detallada
print("\nTabla de comparación de proporciones:")
display(comparison_df)

QUIZÁ NO TENGA MUCHO SENTIDO COMPARAR AQUI LOS DATOS DE SC-RNA-SEQ CON LA DECONVOLUCIÓN SI NO TIENEN LOS MISMOS TIPOS CELULARES

## 2.4. Correlación con el Estadio del Tumor

Investigamos si la proporción de las fracciones celulares (incluyendo 'otherCells')cambia según el estadio patológico del tumor.

In [ ]:
print("\n--- Correlacionando la composición celular con el estadio del tumor ---")

# Seleccionamos fracciones de interés, incluyendo 'otherCells'
fractions_of_interest = ['T cell', 'mononuclear phagocyte', 'malignant cell', 'fibroblast', 'otherCells']

analysis_df['stage_group'] = analysis_df['ajcc_pathologic_stage'].str.extract(r'(Stage [IV]+)')[0]

fig, axes = plt.subplots(1, len(fractions_of_interest), figsize=(25, 6), sharey=True)
fig.suptitle('Proporción de Fracciones Celulares por Estadio del Tumor', fontsize=16)

for i, cell_type in enumerate(fractions_of_interest):
    sns.boxplot(
        data=analysis_df.dropna(subset=['stage_group']),
        x='stage_group',
        y=cell_type,
        ax=axes[i],
        order=['Stage I', 'Stage II', 'Stage III', 'Stage IV']
    )
    axes[i].set_title(f'Fracción de {cell_type}')
    axes[i].set_xlabel('Estadio del Tumor')
    axes[i].set_ylabel('Proporción Estimada')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# 3. Análisis de Supervivencia (Kaplan-Meier)


In [ ]:
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

def plot_kaplan_meier(df, cell_type, ax):
    """Función para generar una curva de Kaplan-Meier para un tipo celular."""
    
    # Crear grupos de "Alta" vs. "Baja" infiltración basado en la mediana
    median_val = df[cell_type].median()
    df['group'] = np.where(df[cell_type] >= median_val, 'Alta', 'Baja')
    
    kmf_alta = KaplanMeierFitter()
    kmf_baja = KaplanMeierFitter()
    
    # Ajustar el modelo a cada grupo
    alta_group = df[df['group'] == 'Alta']
    baja_group = df[df['group'] == 'Baja']
    
    kmf_alta.fit(alta_group['survival_time'], alta_group['event_status'], label=f'Alta {cell_type} (n={len(alta_group)})')
    kmf_baja.fit(baja_group['survival_time'], baja_group['event_status'], label=f'Baja {cell_type} (n={len(baja_group)})')
    
    # Graficar
    kmf_alta.plot_survival_function(ax=ax)
    kmf_baja.plot_survival_function(ax=ax)
    
    # Test estadístico (Log-Rank)
    results = logrank_test(
        alta_group['survival_time'], baja_group['survival_time'],
        event_observed_A=alta_group['event_status'], event_observed_B=baja_group['event_status']
    )
    
    ax.set_title(f'Supervivencia según la fracción de {cell_type}\nLog-Rank p-value: {results.p_value:.3f}')
    ax.set_xlabel('Tiempo (días)')
    ax.set_ylabel('Probabilidad de Supervivencia')
    ax.grid(True)

# Creamos una figura para varias curvas de Kaplan-Meier
fig, axes = plt.subplots(4, 3, figsize=(21, 21))
fig.suptitle('Análisis de Supervivencia de Kaplan-Meier', fontsize=16)

rep_cells = all_cell_fractions

for cell, ax in zip(rep_cells, axes.flatten()):
    plot_kaplan_meier(analysis_df, cell, ax)

for i in range(len(rep_cells), len(axes.flatten())):
    axes.flatten()[i].axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()